<a href="https://colab.research.google.com/github/md-shafiq-cloud-architect/aws-cloud-security-portfolio/blob/main/01_refining_iam_permissions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install cfn-lint --quiet

pipeline_template = """
AWSTemplateFormatVersion: '2010-09-09'
Description: Automated IAM Access Analyzer to EventBridge, Lambda, and SNS Pipeline

Resources:
  # 1. Enable IAM Access Analyzer
  SecurityAccessAnalyzer:
    Type: AWS::AccessAnalyzer::Analyzer
    Properties:
      AnalyzerName: Central-Security-Analyzer
      Type: ACCOUNT

  # 3. Configure SNS Topic for Cloud Admin Alerts
  CloudAdminAlertTopic:
    Type: AWS::SNS::Topic
    Properties:
      TopicName: CloudAdmin-Security-Alerts

  # Alert Processor AWS Lambda Function
  PolicyAlertLambda:
    Type: AWS::Lambda::Function
    Properties:
      FunctionName: ProcessAccessAnalyzerFinding
      Runtime: python3.12
      Handler: index.lambda_handler
      Role: arn:aws:iam::123456789012:role/SecOpsRole
      Code:
        ZipFile: |
          import json
          def lambda_handler(event, context):
              print("Alerting Cloud Admin via SNS:", json.dumps(event))
              return {"status": "Alert Sent"}

  # 2. Configure Amazon EventBridge Rule
  AnalyzerFindingRule:
    Type: AWS::Events::Rule
    Properties:
      Name: Route-IAM-Findings-To-Lambda
      EventPattern:
        source:
          - aws.access-analyzer
        detail-type:
          - Access Analyzer Finding
      Targets:
        - Arn: !GetAtt PolicyAlertLambda.Arn
          Id: TargetLambda
"""

with open("iam-analyzer-pipeline.yaml", "w") as f:
    f.write(pipeline_template.strip())

!cfn-lint iam-analyzer-pipeline.yaml && echo "SUCCESS: Pipeline validated with zero errors!"

SUCCESS: Pipeline validated with zero errors!


In [4]:
import yaml, json

# 1. Create CodeBuild buildspec.yml per Lab Step 8
buildspec_content = """
version: 0.2
phases:
  install:
    runtime-versions:
      python: 3.12
    commands:
      - pip install boto3 cfn-lint
  build:
    commands:
      - echo "Extracting IAM policy into policy.json..."
      - echo "Running IAM Access Analyzer ValidatePolicy..."
      - aws accessanalyzer validate-policy --policy-document file://policy.json --policy-type IDENTITY_POLICY
      - echo "Running CheckAccessNotGranted to block sensitive s3:PutBucketPolicy..."
      - aws accessanalyzer check-access-not-granted --policy-document file://policy.json --access actions=s3:PutBucketPolicy --policy-type IDENTITY_POLICY
"""

with open("buildspec.yml", "w") as f:
    f.write(buildspec_content.strip())

# Validate buildspec.yml syntax
parsed_spec = yaml.safe_load(open("buildspec.yml"))
print("1. buildspec.yml created & validated successfully! Phases:", list(parsed_spec["phases"].keys()))

# 2. Simulate CheckAccessNotGranted blocking s3:PutBucketPolicy
dev_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Action": ["s3:GetObject", "s3:PutBucketPolicy"], "Resource": "*"}]
}
restricted_action = "s3:PutBucketPolicy"
granted_actions = [act for stmt in dev_policy["Statement"] for act in stmt["Action"]]

if restricted_action in granted_actions:
    print(f"2. SECURITY ALERT [FAIL]: Sensitive permission '{restricted_action}' detected! Build blocked.")
else:
    print("2. SECURITY CHECK [PASS]: No restricted permissions found.")

1. buildspec.yml created & validated successfully! Phases: ['install', 'build']
2. SECURITY ALERT [FAIL]: Sensitive permission 's3:PutBucketPolicy' detected! Build blocked.


In [6]:
import json

# 1. Testing SEC203 Role with your exact intentional bad policy from Note Step 9
sec203_test_policy = {
    "Action": "iam:PassRole",
    "Resource": "*",
    "Effect": "allow" # <--- এখানে REPLACE_ME মুছে তোমার নোটের মতো ছোট হাতের allow লেখো
}

print("Testing SEC203 Policy:", json.dumps(sec203_test_policy))

# 2. Simulate LambdaPolicyValidatorFunction (Grammar Check)
if sec203_test_policy["Effect"] != "Allow":
    print("EMAIL 1 SENT -> [Policy Document Check for Policy Validation]: Grammar error! 'Effect' must be 'Allow', found:", sec203_test_policy["Effect"])

# 3. Simulate LambdaCustomPolicyChecks (Critical Permission Check)
if sec203_test_policy["Action"] == "iam:PassRole" and sec203_test_policy["Resource"] == "*":
    print("EMAIL 2 SENT -> [Policy Document Check for Custom Policy Checks]: Critical risk! Wildcard iam:PassRole detected!")

Testing SEC203 Policy: {"Action": "iam:PassRole", "Resource": "*", "Effect": "allow"}
EMAIL 1 SENT -> [Policy Document Check for Policy Validation]: Grammar error! 'Effect' must be 'Allow', found: allow
EMAIL 2 SENT -> [Policy Document Check for Custom Policy Checks]: Critical risk! Wildcard iam:PassRole detected!


In [9]:
import json

rule = {
    "RuleName": "WorkshopUnusedAccess-EventBridgeRule",
    "EventPattern": {
        "source": ["aws.access-analyzer"],
        "detail-type": ["UnusedAccessFinding"]
    },
    "Target": "WorkshopUnusedAccessStack-LambdaUnusedAccess"
}

if rule["EventPattern"]["detail-type"] == ["UnusedAccessFinding"]:
    print("1. Rule Validated:", rule["RuleName"])
    print("2. Triggering Lambda:", rule["Target"])
    print("3. EMAIL SENT -> Subject: 'Unused Permissions Findings'")

1. Rule Validated: WorkshopUnusedAccess-EventBridgeRule
2. Triggering Lambda: WorkshopUnusedAccessStack-LambdaUnusedAccess
3. EMAIL SENT -> Subject: 'Unused Permissions Findings'


In [10]:
import os, subprocess

# 1. Create workshop_repo & initialize real Git repository
os.makedirs("workshop_repo", exist_ok=True)
os.chdir("workshop_repo")
subprocess.run(["git", "init", "-q"])
subprocess.run(["git", "config", "user.email", "devops@company.com"])
subprocess.run(["git", "config", "user.name", "DevOpsEngineer"])

# 2. Step 12 (Part 2): Good Policy in stack_workshop.py -> Pipeline PASS
good_code = '''
POLICY = {
    "Action": "sns:Publish",
    "Resource": "arn:aws:sns:us-east-1:123456789012:IAMAccessAnalyzerFindingNotifications"
}
'''
with open("stack_workshop.py", "w") as f:
    f.write(good_code)

subprocess.run(["git", "add", "--all"])
subprocess.run(["git", "commit", "-q", "-m", "add permissions for lambda"])
print("1. [GOOD PUSH] Commit done! Pipeline Status: SUCCESS -> Email: 'Pipeline Build Result (PASS)'")

# 3. Step 12 (Part 3): Bad Policy (Wildcard *) in stack_workshop.py -> Pipeline FAIL
bad_code = '''
POLICY = {
    "Action": "sns:*",
    "Resource": "*"
}
'''
with open("stack_workshop.py", "w") as f:
    f.write(bad_code)

subprocess.run(["git", "add", "--all"])
subprocess.run(["git", "commit", "-q", "-m", "add overly permissive wildcard"])
if "*" in bad_code:
    print("2. [BAD PUSH] Commit done! Custom Policy Check Status: FAIL -> Overly permissive wildcard blocked!")

1. [GOOD PUSH] Commit done! Pipeline Status: SUCCESS -> Email: 'Pipeline Build Result (PASS)'
2. [BAD PUSH] Commit done! Custom Policy Check Status: FAIL -> Overly permissive wildcard blocked!


In [11]:
import os, shutil

# Step 14: Clean up lab workspace & resources per Lab Guide
os.chdir("/content")
if os.path.exists("workshop_repo"):
    shutil.rmtree("workshop_repo")

print("CLEANUP COMPLETE: Temporary workshop_repo and mock resources removed safely!")

CLEANUP COMPLETE: Temporary workshop_repo and mock resources removed safely!
